# Training logistic regression models

In [25]:
import json
from collections import Counter
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_recall_fscore_support
import joblib


data_path = "../data"

with open(f"{data_path}/train_classic_ml_set.json", "r") as f:
    train_classic_ml_set = json.load(f)

with open(f"{data_path}/test_set.json", "r") as f:
    test_set = json.load(f)

BoW_MODEL_PATH = "../models/LR_BoW/3H1L_BoW_LR_model.pkl"
BoW_VECTORIZER_PATH = "../models/LR_BoW/3H1L_BoW_vectorizer.pkl"

### Transform data into appropriate format

Create Python list of dicts

In [33]:
def lr_transform(train_or_test_set):

    modified_set = []

    for item in train_or_test_set:
        date = item["date"]
        answer = item["answer"]
        for index, headline in enumerate(item["headlines"]):
            if index == answer:
                fake = 1
            else:
                fake = 0
            entry = {
                "Date": date,
                "Headline": headline,
                "Fake": fake
            }
            modified_set.append(entry)

    return modified_set

train_set_lr_ready = lr_transform(train_classic_ml_set)
test_set_lr_ready = lr_transform(test_set)


Transform into dataframe and create X and y classes

In [ ]:
train_df = pd.DataFrame(train_set_lr_ready)

X_train_text = train_df["Headline"]

y_train = train_df["Fake"]

test_df = pd.DataFrame(test_set_lr_ready)

X_test_text = test_df["Headline"]

y_test = test_df["Fake"] # May not actually use this given quiz level accuracy is primary metric

### Train BoW TF-IDF LR model 

Model training or loading

In [35]:
try:
    clf = joblib.load(BoW_MODEL_PATH)
    vectorizer = joblib.load(BoW_VECTORIZER_PATH)
    print("Loaded model and vectorizer from disk.")

except FileNotFoundError:

    print("No saved model found, training a new one...")

    # Create TF-IDF vectorizer
    vectorizer = TfidfVectorizer(
        max_features=5000,  # cap number of features for speed
        ngram_range=(1, 2)  # include single words and short phrases
    )

    # Fit on training data, transform both train and test sets
    X_train = vectorizer.fit_transform(X_train_text)
    X_test = vectorizer.transform(X_test_text)

    # Initialise model
    clf = LogisticRegression(max_iter=1000, random_state=42)

    # Train model
    clf.fit(X_train, y_train)

    joblib.dump(clf, BoW_MODEL_PATH)
    joblib.dump(vectorizer, BoW_VECTORIZER_PATH)
    print("Model and vectorizer saved.")

else:
    X_train = vectorizer.transform(X_train_text)
    X_test = vectorizer.transform(X_test_text)

print("TF-IDF matrix shape (train):", X_train.shape)
print("TF-IDF matrix shape (test):", X_test.shape)

No saved model found, training a new one...
Model and vectorizer saved.
TF-IDF matrix shape (train): (1080, 5000)
TF-IDF matrix shape (test): (268, 5000)


Evaluate

In [38]:
y_pred = clf.predict(X_test)
y_fake_prob = clf.predict_proba(X_test)[:, 1]

In [57]:
test_df["Fake Prob"] = y_fake_prob
test_by_quiz_df = test_df.groupby("Date")

correct = 0

for date, quiz_df in test_by_quiz_df:
    predicted_index = quiz_df["Fake Prob"].idxmax()
    actual_index = quiz_df["Fake"].idxmax()

    if predicted_index == actual_index:
        correct += 1

accuracy = correct / len(test_set)

print(f"Quiz-level accuracy: {accuracy:.2%}")

Quiz-level accuracy: 43.28%
